1) Import everything we need

In [19]:

# Modules for model workflows and transformer building
import numpy as np
import random
import torch
from torch import nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoderLayer, TransformerDecoder
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset
from torch.optim import Adam
from math import floor

# Modules for data generation
from scipy.signal import cont2discrete

# Modules for some custom loss function
from torch.linalg import inv

2) Setup seed for reproducibility

In [20]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

3) Define transformer class

In [21]:
class TransformerAutoencoder(nn.Module):
    def __init__(self, 
                 encoder_input_dim, 
                 decoder_input_dim, 
                 hidden_dim,
                 num_heads, 
                 encoder_embedding_dim, 
                 decoder_embedding_dim,
                 num_layers, 
                 dropout):
        super(TransformerAutoencoder, self).__init__()
        self.encoder_input_dim = encoder_input_dim
        self.decoder_input_dim = decoder_input_dim
        self.encoder_embedding_dim = encoder_embedding_dim
        self.decoder_embedding_dim = decoder_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout

        # Encoder Embedding
        self.encoder_embedding = nn.Linear(self.encoder_input_dim, 
                                           self.encoder_embedding_dim)

        # Encoder
        self.encoder_layer = TransformerEncoderLayer(d_model=self.encoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.encoder = TransformerEncoder(self.encoder_layer,
                                          num_layers=self.num_layers)

        # Decoder Embedding
        self.decoder_embedding = nn.Linear(self.decoder_input_dim, 
                                           self.decoder_embedding_dim)

        # Decoder
        self.decoder_layer = TransformerDecoderLayer(d_model=self.decoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.decoder = TransformerDecoder(self.decoder_layer, 
                                          num_layers=self.num_layers)

        # Final output layer
        self.out = nn.Linear(self.decoder_embedding_dim,
                             self.decoder_input_dim)

    def forward(self, inputs, targets):        
        # Encode the input
        encoded_input = self.encoder(self.encoder_embedding(inputs))
        
        # Decode the target
        decoder_input = self.decoder_embedding(targets)
        decoded_target = self.decoder(decoder_input, encoded_input)        
        
        # Apply the final output layer
        target_output = self.out(decoded_target)
        return target_output

4) Data for our experiments: Lorrentz data, Linear springs data, Kuramoto Shivasinsky

a) Lorrentz Data: 
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [22]:
import math
def generate_lorenz_data_torch(seq_length=200, num_records=50, dt=0.01):
    dim_x = 3  # state dimension
    dim_y = 1  # observation dimension

    # Lorenz system matrix C 
    C = torch.tensor([[-10, 10, 0],
                        [28, -1, 0],
                        [0, 0, -8/3]], dtype=torch.float32)

    # Observation matrix H: only considers first two states
    H = torch.tensor([[1.0, 1.0, 0.0]], dtype=torch.float32).reshape(1, dim_y, dim_x)

    # Define state transition function based on matrix exponential 
    def f(x):
        BX = torch.zeros(x.shape[0], dim_x, dim_x) # [batch_size, dim_x, dim_x] 
        BX[:, 1, 0] = -x[:, 2, 0]
        BX[:, 2, 0] = x[:, 1, 0]
        A = BX + C  

        # Taylor Expansion for F   
        F = torch.eye(dim_x).unsqueeze(0).repeat(x.shape[0], 1, 1) # [batch_size, dim_x, dim_x] identity matrix
        for j in range(1, 10): 
            F += torch.matrix_power(A * dt, j) / math.factorial(j)
        return torch.bmm(F, x)


    Q_val = 0.01  # process noise variance
    R_val = 0.01  # observation noise variance
    Q = Q_val * torch.eye(dim_x)
    R = R_val * torch.eye(dim_y)

    X_data = torch.empty((num_records, seq_length, dim_x, 1))  # state data
    Y_data = torch.empty((num_records, seq_length, dim_y, 1))  # observation data

    # Generate data for each trajectory (sequence)
    for i in range(num_records):
        # Initialize initial state randomly between [-2, 2] plus small random noise
        x = torch.randn(dim_x, 1).uniform_(-2, 2)
        x += torch.randn_like(x) * 0.01
        x = x.unsqueeze(0)  # Add batch dimension
        y = torch.bmm(H, x) + torch.randn(1, dim_y, 1) * torch.sqrt(R) # Generate initial observation with observation noise

        # Store initial state and observation
        X_data[i, 0] = x.squeeze(0)
        Y_data[i, 0] = y.squeeze(0)

        # Generate the remaining sequence
        for t in range(1, seq_length):
            x = f(x)  # Apply state transition
            # Add process noise: Q 
            L = torch.linalg.cholesky(Q)
            x += torch.matmul(L, torch.randn_like(x))
            # Generate noisy observation
            y = torch.bmm(H, x) + torch.randn(1, dim_y, 1) * math.sqrt(R[0, 0])
            # Store state and observation
            X_data[i, t] = x.squeeze(0)
            Y_data[i, t] = y.squeeze(0)
    return X_data, Y_data, H, Q_val, R_val

X_data, Y_data, H_tensor, _, _ =generate_lorenz_data_torch(seq_length=1000, num_records=100, dt=0.01)

# State matrix for Lorentz
C = torch.tensor([[-10, 10,    0],
                  [ 28, -1,    0],
                  [  0,  0, -8/3]]).float()
# define F matrix
m  = 3
J = 10
delta_t = 0.01
def f_jacob(x):
    BX = torch.zeros([x.shape[0], m, m], dtype=torch.float32, device=x.device)
    BX[:, 1, 0] = -x[:, 2, 0]
    BX[:, 2, 0] = x[:, 1, 0]
    
    A = BX + C.to(x.device)
    A = torch.clamp(A, -30.0, 30.0)  
    F = torch.eye(m, device=x.device).reshape(1, m, m).repeat(x.shape[0], 1, 1)
    for j in range(1, J+1):
        F_add = torch.matrix_power(A * delta_t, j) / math.factorial(j)
        F = F + F_add
    F = torch.clamp(F, -1e3, 1e3)  
    return torch.bmm(F, x)

# define H matrix
H = torch.tensor(H_tensor, dtype=torch.float32)
def h(x):
    return torch.matmul(H_tensor.repeat(x.shape[0], 1, 1), x)  



KeyboardInterrupt: 

In [ ]:


allInputs = Y_data.squeeze(-1)
allTargets = X_data.squeeze(-1)

print(allInputs.shape)
print(allTargets.shape)
numSequence = allInputs.shape[0]
encoder_input_dim = allInputs.shape[2]
decoder_input_dim = allTargets.shape[2]

torch.Size([100, 1000, 1])
torch.Size([100, 1000, 3])


b) Linear springs data:
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [ ]:
# rawData = np.load("../Data/linear_springs_data.npz")
# allInputs = torch.Tensor(rawData["Y_data"])
# allTargets = torch.Tensor(rawData["X_data"])
# A = torch.Tensor(rawData["A_matrix"])
# H = torch.Tensor(rawData["H_matrix"])
# Q = torch.Tensor(rawData["Q_matrix"])
# R = torch.Tensor(rawData["R_matrix"])
# P = torch.Tensor(rawData["P_matrix"])
# def f(x): return A @ x
# def h(x): return H @ x
# numSequence = allInputs.shape[0]
# encoder_input_dim = allInputs.shape[2]
# decoder_input_dim = allTargets.shape[2]

5) Split the data into training, validation and testing

In [ ]:
trainPercent = 0.7
testPercent = 0.15
validatePercent = 0.15
lenTrainingData = floor(trainPercent*numSequence)
lenTestingData = floor(testPercent*numSequence)
lenValidatingData = numSequence-lenTestingData-lenTrainingData

print("## Split the data:\n")
print(f"Training data set size   : {lenTrainingData}\n",
      f"Validation data set size : {lenTestingData}\n",
      f"Test data set size       : {lenValidatingData}")

trainingTensorDataSet = TensorDataset(allInputs[0:lenTestingData-1],
                                      allTargets[0:lenTestingData-1])
testingTensorDataSet = TensorDataset(allInputs[lenTestingData:lenTestingData+lenTrainingData-1],
                                     allTargets[lenTestingData:lenTestingData+lenTrainingData-1])
validatingTensorDataSet = TensorDataset(allInputs[lenTestingData+lenTestingData:],
                                        allTargets[lenTestingData+lenTestingData:])

## Split the data:

Training data set size   : 70
 Validation data set size : 15
 Test data set size       : 15


6) Define MSE loss function evaluator. Do back propagation as we evaluate the loss for every dataset we train

In [ ]:
def evaluateMSELoss(model, optimizer, train_dataloader):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        loss = criterion(outputs, targets)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

def evaluateCustomLoss1(model, optimizer, train_dataloader, alpha):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        h_of_outputs = 0.0*inputs 
        loss = alpha * criterion(outputs, targets) + (1.0 - alpha)*criterion(inputs[0,:,:], h(outputs[0,:,:].T).T)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

7) Define network training and validating parameters

In [49]:
alpha = 0.8
num_layers = 4
num_epochs = 100
hidden_dim = 64
num_heads = 16
encoder_embedding_dim = 16
decoder_embedding_dim = 16
learn_rate = 0.0001
weight_decay = 0.005
dropout = 0.05

trainDataSetLoader =  DataLoader(trainingTensorDataSet, batch_size = 1)
validationDataSetLoader = DataLoader(validatingTensorDataSet)
testDataSetLoader = DataLoader(testingTensorDataSet)

# Data dict to store best results
best_result = dict({
    'model':[],
    'learn_rate':[],
    'num_epochs':[],
    'encoder_embedding_dim':[],
    'decoder_embedding_dim':[],
    'hidden_dim':[],
    'num_heads':[],
    'weight_decay':[],
    'dropout':[],
    'avg_training_loss':[],
    'best_validation_loss':[]
})

8) Define model and optimizer

In [47]:
model = TransformerAutoencoder(encoder_input_dim, 
                               decoder_input_dim,
                               hidden_dim, 
                               num_heads, 
                               encoder_embedding_dim,
                               decoder_embedding_dim, 
                               num_layers, 
                               dropout)
optimizer = Adam(model.parameters(), 
                 lr=learn_rate, 
                 weight_decay=weight_decay)

In [ ]:
best_validation_loss = 1000000.
avg_training_loss = 0.
avg_validation_loss = 0.

for epoch in range(num_epochs):
    print(f"EPOCH NUMBER: {epoch+1}")
    model.train(True)
    avg_training_loss = evaluateCustomLoss1(model, optimizer, trainDataSetLoader, alpha)
    model.eval()
    
    sum_validation_loss = 0.0

    with torch.no_grad():
        for i, batch in enumerate(validationDataSetLoader):
            inputs, targets = batch
            outputs = model(inputs, targets)
            criterion = nn.MSELoss()
            val_loss = criterion(outputs, targets)
            sum_validation_loss += val_loss.item()
    avg_validation_loss = sum_validation_loss / len(validationDataSetLoader)
    
    print(f"AVERAGE TRAINING LOSS  : {avg_training_loss}\nAVERAGE VALIDATION LOSS: {avg_validation_loss}")

    if avg_validation_loss < best_validation_loss:
        best_validation_loss = avg_validation_loss
        print(f"BEST VALIDATION LOSS: {best_validation_loss} at EPOCH {epoch+1}")
        best_result['model']=model.state_dict()
        best_result['learn_rate']=learn_rate
        best_result['encoder_embedding_dim']=encoder_embedding_dim
        best_result['decoder_embedding_dim'] =decoder_embedding_dim
        best_result['num_epochs']=num_epochs
        best_result['hidden_dim']=hidden_dim
        best_result['num_heads']=num_heads
        best_result['weight_decay']=weight_decay
        best_result['dropout']=dropout
        best_result['avg_training_loss']=avg_training_loss
        best_result['best_validation_loss']=best_validation_loss

EPOCH NUMBER: 1
AVERAGE TRAINING LOSS  : 230.62613677978516
AVERAGE VALIDATION LOSS: 228.87035042898995
BEST VALIDATION LOSS: 228.87035042898995 at EPOCH 1
EPOCH NUMBER: 2
AVERAGE TRAINING LOSS  : 227.1104027884347
AVERAGE VALIDATION LOSS: 225.79423980712892
BEST VALIDATION LOSS: 225.79423980712892 at EPOCH 2
EPOCH NUMBER: 3
AVERAGE TRAINING LOSS  : 224.21992710658483
AVERAGE VALIDATION LOSS: 222.9808125087193
BEST VALIDATION LOSS: 222.9808125087193 at EPOCH 3
EPOCH NUMBER: 4
AVERAGE TRAINING LOSS  : 221.4340874808175
AVERAGE VALIDATION LOSS: 220.2079347882952
BEST VALIDATION LOSS: 220.2079347882952 at EPOCH 4
EPOCH NUMBER: 5


9) Test the unseen data

In [50]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'lorenz_checkpoint_latest.pth')
checkpoint = torch.load('lorenz_checkpoint_latest.pth')
# Restore model & optimizer
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1  # Resume from next epoch

/tmp/ipykernel_366804/1784279278.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('lorenz_checkpoint_latest.pth')


In [43]:
model.eval()
sum_testing_loss = 0.0
total_seqs = 0

criterion = nn.MSELoss()
for i, batch in enumerate(testDataSetLoader):
    inputs, targets = batch
    outputs = model(inputs, targets)
    criterion = nn.MSELoss()
    test_loss = criterion(outputs, targets)
    sum_testing_loss += test_loss.item()
avg_testing_loss = sum_testing_loss / len(testDataSetLoader)
print(f"[MSE] = {avg_testing_loss:.6f}")

[MSE] = 135.458735


In [ ]:
print(x_true.shape)

torch.Size([1, 100, 20])
